In [17]:
import pandas as pd
import torch.nn as nn
import torch.optim as optim


from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

In [18]:
data = load_breast_cancer()

In [19]:
x = data.data
y = data.target

In [20]:
x

array([[1.799e+01, 1.038e+01, 1.228e+02, ..., 2.654e-01, 4.601e-01,
        1.189e-01],
       [2.057e+01, 1.777e+01, 1.329e+02, ..., 1.860e-01, 2.750e-01,
        8.902e-02],
       [1.969e+01, 2.125e+01, 1.300e+02, ..., 2.430e-01, 3.613e-01,
        8.758e-02],
       ...,
       [1.660e+01, 2.808e+01, 1.083e+02, ..., 1.418e-01, 2.218e-01,
        7.820e-02],
       [2.060e+01, 2.933e+01, 1.401e+02, ..., 2.650e-01, 4.087e-01,
        1.240e-01],
       [7.760e+00, 2.454e+01, 4.792e+01, ..., 0.000e+00, 2.871e-01,
        7.039e-02]], shape=(569, 30))

In [21]:
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
       1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
       1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0,
       0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1,
       1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0,

In [22]:
x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

In [23]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [24]:
from torch.utils.data import Dataset, DataLoader
import torch

class CustomDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)

        self.labels = torch.tensor(labels, dtype=torch.float32).view(-1, 1)


    def __len__(self):
        return len(self.features)    
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]        

In [25]:
train_dataset = CustomDataset(x_train, y_train)
test_dataset = CustomDataset(x_test, y_test)

In [26]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, pin_memory=True)

In [27]:
class MyNN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 12),
            nn.ReLU(),

            nn.Linear(12, 1),
            

        )

    def forward(self, features):
        return self.network(features)



In [28]:
lr = 0.001
loss = nn.BCEWithLogitsLoss()
epochs = 50

In [29]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [30]:
model = MyNN(x_train.shape[1])

model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=lr)

In [31]:
for epoch in range(epochs):
    total_loss = 0
    model.train()
    for batch_features, batch_labels in train_loader:
        batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)

        y_pred = model(batch_features)

        loss_item = loss(y_pred, batch_labels)

        optimizer.zero_grad()

        loss_item.backward()

        optimizer.step()

        total_loss += loss_item.item()
    print(total_loss/len(train_loader))    

0.6973707993825277
0.6956059217453003
0.6939997990926107
0.6954612692197164
0.6941539883613587
0.6935458024342854
0.6912348866462708
0.6916572014490764
0.6905080080032349
0.6891480882962545
0.6893195668856303
0.6887836297353108
0.6881064295768737
0.6867130676905314
0.6875160018603007
0.6846667130788168
0.6852688471476237
0.6838493188222249
0.6824363271395365
0.6839919447898865
0.6839284459749858
0.683111306031545
0.6791727264722188
0.6789415756861369
0.6781157970428466
0.6781870206197103
0.6780762116114298
0.6765074968338013
0.675720735390981
0.6749969800313314
0.6766944368680318
0.6732882738113404
0.6735530336697896
0.6723114172617595
0.6718199133872986
0.6707425594329834
0.668302611509959
0.6669138749440511
0.6686870177586873
0.6698994517326355
0.6677523771921794
0.6628816445668538
0.6644271771113078
0.6642991622289022
0.6626293142636617
0.661530343691508
0.6589692274729411
0.6603194276491801
0.6580795208613078
0.6578590035438537


In [32]:
# %%
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch_features, batch_labels in test_loader:

        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)

        # logits
        outputs = model(batch_features)

        # logits -> probabilities
        probs = torch.sigmoid(outputs)

        # probabilities -> 0 or 1
        predictions = (probs >= 0.5).float()

        correct += (predictions == batch_labels).sum().item()

        total += batch_labels.size(0)

accuracy = correct / total

print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.6228
